In [77]:
import re
import sys
import statistics
import altair as alt
import networkx as nx
import pandas as pd
from collections import defaultdict

In [78]:
lung = '/Data/data_mastro/trees-lung.txt'
aml = '/Data/data_mastro/trees-aml.txt'

In [79]:
def analyze_data(file):
    data_dict = {'distinct': [], 'clusters': [], 'edges': []}
    mutation_count = defaultdict(lambda: 0)
    with open(file, 'r') as f:
        for i, line in enumerate(f):
            line = line.strip()
            if not line:
                continue

            # Split keywords by all separators
            keywords = re.split(r'-/-|-\?-|->-|\s', line)
            distinct_keywords = set(keywords)
            distinct_keywords.discard('')

            for keyword in distinct_keywords:
                mutation_count[keyword] += 1

            # Build connected componets from clusters to count them properly
            G = nx.Graph()
            for pair in line.split(' '):
                if '-?-' in pair:
                    a, b = pair.split('-?-')
                    G.add_edge(a, b)
            clusters = [c for c in nx.connected_components(G)]

            # Count direct edges
            arrow_count = len(re.findall(r'->-', line))

            # print(f"Line {i}: distinct={len(distinct_keywords)}, clusters={len(clusters)}, arrows={arrow_count}")
            data_dict['distinct'].append(len(distinct_keywords))
            data_dict['clusters'].append(len(clusters))
            data_dict['edges'].append(arrow_count)

    return data_dict, mutation_count

In [90]:
stats, mutations = analyze_data(lung)
df = {'Gene': [], 'Count': []}
for key, value in mutations.items():
    df['Gene'].append(key)
    df['Count'].append(value)
df = pd.DataFrame.from_dict(df)
df.sort_values(by='Count', ascending=False, inplace=True)

chart = alt.Chart(df[df['Count'] >= 5], title='Genes mutated in at least five NSCLC patients').mark_bar(color='#98b2f3').encode(
    x=alt.X('Gene:O', sort='-y'),
    y='Count'
).properties(
    height=200,
    width=350
)
chart.show()
chart.save('result_data/lung_gene_counts_geq_5_patients.html')

alt.Chart(...)

In [81]:
out_file_lung = 'result_data/lung_data_stats.csv'
with open(out_file_lung, 'w') as f:
    f.write(',Mean,Median,Maximum\n')
    f.write('Mutations,' + ','.join([str(statistics.mean(stats['distinct'])), str(statistics.median(stats['distinct'])), str(max(stats['distinct']))]) + '\n')
    f.write('Clusters,' + ','.join([str(statistics.mean(stats['clusters'])), str(statistics.median(stats['clusters'])), str(max(stats['clusters']))]) + '\n')
    f.write('Edges,' + ','.join([str(statistics.mean(stats['edges'])), str(statistics.median(stats['edges'])), str(max(stats['edges']))]) + '\n')

In [88]:
stats, mutations = analyze_data(aml)
df = {'Gene': [], 'Count': []}
for key, value in mutations.items():
    df['Gene'].append(key)
    df['Count'].append(value)
df = pd.DataFrame.from_dict(df)
chart = alt.Chart(df, title='Genes mutated in AML patients').mark_bar(color='#98b2f3').encode(
    x=alt.X('Gene:O', sort='-y'),
    y='Count'
).properties(
    height=200,
    width=350
)
chart.show()
chart.save('result_data/aml_gene_counts.html')

out_file_aml = 'result_data/aml_data_stats.csv'
with open(out_file_aml, 'w') as f:
    f.write(',Mean,Median,Maximum\n')
    f.write('Mutations,' + ','.join([str(statistics.mean(stats['distinct'])), str(statistics.median(stats['distinct'])), str(max(stats['distinct']))]) + '\n')
    f.write('Clusters,' + ','.join([str(statistics.mean(stats['clusters'])), str(statistics.median(stats['clusters'])), str(max(stats['clusters']))]) + '\n')
    f.write('Edges,' + ','.join([str(statistics.mean(stats['edges'])), str(statistics.median(stats['edges'])), str(max(stats['edges']))]) + '\n')

alt.Chart(...)